# BI Intern Assignment

## Setup
The sheets have a '💡' row at the end; I drop the rows with no key so they don't pollute the data. SQL is run on an SQLite database built from the same tables.

In [ ]:
import pandas as pd, sqlite3
FILE = "bi_intern_assignment.xlsx"
orders    = pd.read_excel(FILE, "📊 orders").dropna(subset=["customer_id"])
customers = pd.read_excel(FILE, "👤 customers").dropna(subset=["customer_name"])
catalog   = pd.read_excel(FILE, "🗂 product_catalog").dropna(subset=["product_name"])
pricing   = pd.read_excel(FILE, "💰 product_pricing").dropna(subset=["channel"])
orders["order_id"] = orders["order_id"].astype(int); orders["quantity"] = orders["quantity"].astype(int)

con = sqlite3.connect(":memory:")
orders.to_sql("orders", con, index=False); customers.to_sql("customers", con, index=False)
catalog.to_sql("product_catalog", con, index=False); pricing.to_sql("product_pricing", con, index=False)
def sql(q): return pd.read_sql_query(q, con)

# data quality check before proceeding with the tasks
print({n: len(d) for n, d in dict(orders=orders, customers=customers, catalog=catalog, pricing=pricing).items()})
print("order_id unique:", orders.order_id.is_unique, "| orders w/o customer:", (~orders.customer_id.isin(customers.customer_id)).sum())
print("orders w/o (product, channel) price row:", len(orders.merge(pricing, on=["product_name","channel"], how="left").query("discount_pct.isna()")))

{'orders': 50, 'customers': 26, 'catalog': 18, 'pricing': 90}
order_id unique: True | orders w/o customer: 0
orders w/o (product, channel) price row: 0


## SQL Task
### QS1 — total_orders and avg_MRP per customer_segment
`orders` → `customers` is a single-key join on `customer_id`. `total_orders` = number of order rows (not units).

In [4]:
sql("""
SELECT c.customer_segment,
       COUNT(*)            AS total_orders,
       ROUND(AVG(o.MRP),2) AS avg_MRP
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.customer_segment
ORDER BY total_orders DESC
""")

,customer_segment,total_orders,avg_MRP
0,Gold,22,1327.64
1,Silver,18,1282.11
2,Bronze,10,1282.90


### QS2 — total_revenue per product_name
Discount depends on **both** product and channel, so the join to `product_pricing` is on `(product_name, channel)`.

`revenue = MRP × quantity × (1 − discount_pct/100)`

In [5]:
sql("""
SELECT o.product_name,
       ROUND(SUM(o.MRP * o.quantity * (1 - p.discount_pct/100.0)), 2) AS total_revenue
FROM orders o
JOIN product_pricing p
  ON p.product_name = o.product_name AND p.channel = o.channel
GROUP BY o.product_name
ORDER BY total_revenue DESC
""")

,product_name,total_revenue
0,Vitamin C Serum,30730.35
1,Collagen Supplement,20429.18
2,Conditioner,14298.75
3,Ashwagandha,14100.00
4,Night Cream,13540.80
5,Moisturizer,13172.60
6,Hair Mask,12145.20
7,Vitamin D3,10751.58
8,Eyeliner,10709.26
9,Probiotic Blend,10605.76


### QS3 — which channel has given the highest discount_pct (for orders made)?
Only channel/product combinations that actually appear in orders are considered. The first query ranks channels by the highest discount_pct applied on any order; the second shows the average and the order-weighted view as supporting evidence.

In [6]:
display(sql("""
SELECT o.channel,
       MAX(p.discount_pct)           AS max_discount_pct,
       ROUND(AVG(p.discount_pct),2)  AS avg_discount_pct,
       COUNT(*)                      AS orders
FROM orders o
JOIN product_pricing p ON p.product_name = o.product_name AND p.channel = o.channel
GROUP BY o.channel
ORDER BY max_discount_pct DESC, avg_discount_pct DESC
"""))
sql("""
SELECT o.channel
FROM orders o JOIN product_pricing p ON p.product_name = o.product_name AND p.channel = o.channel
GROUP BY o.channel ORDER BY MAX(p.discount_pct) DESC, AVG(p.discount_pct) DESC LIMIT 1
""")

,channel,max_discount_pct,avg_discount_pct,orders
0,Amazon,22.0,17.43,7
1,Instagram,15.0,11.36,11
2,Website,12.0,9.25,8
3,WhatsApp,10.0,7.11,9
4,App,10.0,5.53,15


,channel
0,Amazon


## Python Task
### QP1 — Category | Total Revenue | Orders | Avg Order Value
Same revenue definition as QS2 (net of channel discount). Avg Order Value = Total Revenue ÷ Orders.

In [7]:
m = orders.merge(pricing, on=["product_name","channel"], how="left", validate="m:1")
m["revenue"] = m["MRP"] * m["quantity"] * (1 - m["discount_pct"]/100)

qp1 = (m.groupby("category")
         .agg(**{"Total Revenue": ("revenue","sum"), "Orders": ("order_id","nunique")})
         .assign(**{"Avg Order Value": lambda d: d["Total Revenue"]/d["Orders"]})
         .round(2).sort_values("Total Revenue", ascending=False).reset_index()
         .rename(columns={"category":"Category"}))
qp1

,Category,Total Revenue,Orders,Avg Order Value
0,Wellness,65337.06,16,4083.57
1,Skincare,64722.59,15,4314.84
2,Haircare,38013.82,14,2715.27
3,Makeup,15283.39,5,3056.68


In [8]:
# cross-check against SQL
chk = sql("""SELECT o.category, SUM(o.MRP*o.quantity*(1-p.discount_pct/100.0)) rev, COUNT(*) n
FROM orders o JOIN product_pricing p ON p.product_name=o.product_name AND p.channel=o.channel GROUP BY o.category""").set_index("category")
assert abs(chk.rev.sum() - qp1["Total Revenue"].sum()) < 0.5 and chk.n.sum() == qp1.Orders.sum()
print("OK — total revenue", round(qp1["Total Revenue"].sum(),2), "| orders", qp1.Orders.sum())

OK — total revenue 183356.86 | orders 50


> Note: the cover sheet mentions *2 python tasks* but the Python tab contains only one (QP1), so only QP1 is answered.

## LLM-Powered HTML Dashboard
### 1) Initial prompt given to the LLM
> You are a senior BI developer. Build a **single-page HTML dashboard** for a D2C beauty & wellness brand's **channel + product performance**. It must run in any browser with **no server and no external dependencies** (one self-contained .html file, inline CSS/JS/SVG, data embedded as JSON).
>
> **Data (embedded, 50 orders, 2024):** order_id, order_date, customer_segment (Gold/Silver/Bronze), category, product_name, quantity, channel (App, Website, Instagram, WhatsApp, Amazon), MRP, discount_pct (joined on product_name + channel), cost_price, is_hero_product.
>
> **Definitions:** net revenue = MRP x quantity x (1 - discount_pct/100); gross profit = net revenue - cost_price x quantity; AOV = net revenue / orders.
>
> **Include:** (1) KPI tiles: net revenue, orders, AOV, gross margin, % given away as discount. (2) Filters for channel, category and segment that update everything. (3) Net revenue by channel with share and average discount. (4) Monthly revenue trend with hover tooltips. (5) Net revenue by product, toggle to gross profit, hero products highlighted. (6) Category table: Category | Total Revenue | Orders | Avg Order Value. (7) Channel table with average discount, margin and a Healthy/Thin flag. (8) Auto-generated key takeaways.
>
> **Design:** clean, executive-friendly, readable on phone, supports light and dark mode, one accent colour, no chart libraries.
### 2) Dashboard link
[Open the dashboard](https://claude.ai/artifact/CgyBa9993yR9zRkmCHhKjB) (single self-contained HTML file; the same file is also saved as `dashboard.html` alongside this notebook).